# ARC-v0.37.1 — HotpotQA Native-Gold Downstream QA Benchmark Freeze
## Transport-Fixed Official→Pinned-Mirror Fallback

## Scientific role

This is a **new benchmark**, not a repair or continuation of ARC-v0.36.

The benchmark is chosen because HotpotQA's official development data natively contains:

- a stable question ID (`_id`);
- the natural-language `question`;
- the gold `answer`;
- question `type` and `level`;
- supporting-fact supervision.

BEIR HotpotQA provides the global 5.23M-document retrieval corpus and BEIR qrels. The key preflight test is whether the **BEIR HotpotQA test query IDs correspond exactly to the official 7,405-question HotpotQA development set**. No answer generation or mechanism outcome is permitted until that equality is verified.

## Why this benchmark

This avoids the ARC-v0.36 failure mode: answer aliases are not reconstructed from a different Natural Questions resplit. The official HotpotQA QA example itself supplies the gold answer, and the shared `_id` is used for alignment rather than fuzzy question matching.

## Prospective downstream experiment

After preflight passes, ARC-v0.37 will freeze:

- dataset: BEIR HotpotQA global retrieval corpus;
- QA gold: official HotpotQA dev distractor answers;
- encoder: `thenlper/gte-small`;
- representation mechanism: IVF-PQ32 @ nprobe64;
- search-effort mechanism: IVF-SQ8 @ FIT-selected nprobe;
- shared high branch: IVF-SQ8 @ nprobe64;
- search grid: `[1,2,4,8,16,32]`;
- one-shot utility: nDCG@10;
- feedback operator: anchored centroid;
- H=4;
- same eight structural policies used in the prior mechanism audit;
- answerer: `Qwen/Qwen2.5-3B-Instruct`;
- deterministic decoding;
- terminal evidence: top-5 retrieved passages;
- answer metrics: official-style normalized EM and token F1;
- primary downstream estimand: feedback-added F1 mechanism contrast;
- independent sampling unit: query;
- 10,000 paired-query bootstrap replicates.

This notebook performs **only source validation, split construction, membership freeze, and protocol freeze**. It deliberately stops before embeddings, ANN calibration, trajectories, or answer generation.

### v0.37.1 transport-only repair

ARC-v0.37 stopped when the official CMU host timed out, before any retrieval, ANN,
trajectory, or answer outcome existed. v0.37.1 changes only acquisition transport:
try the official CMU URL first, then fall back to a pinned RAGLAB/Hugging Face copy.
The fallback bytes must match SHA256
`4e9ecb5c8d3b719f624d66b60f8d56bf227f03914f5f0753d6fa1b359d7104ea`.

Regardless of transport, the benchmark is accepted only if all 7,405 BEIR test IDs
and normalized question texts match the acquired HotpotQA dev distractor examples.


In [ ]:
# Cell 1 — Environment and constants
from pathlib import Path
from collections import defaultdict
import hashlib, json, os, re, unicodedata, zipfile, requests

SEED = 20260830
SPLIT_SALT = "ARC-v0.37-HOTPOTQA-NATIVE-GOLD-SPLIT-v1"
MAIN_SALT = "ARC-v0.37-HOTPOTQA-NATIVE-GOLD-MAIN500-v1"

N_OFFICIAL_EXPECTED = 7405
N_MAIN = 500

ROOT = Path("/content/arc-v037-hotpot-native-gold")
ROOT.mkdir(parents=True, exist_ok=True)
OUT = ROOT / "out"
OUT.mkdir(parents=True, exist_ok=True)

BEIR_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/hotpotqa.zip"
BEIR_ZIP = ROOT / "hotpotqa.zip"
BEIR_DIR = ROOT / "hotpotqa"

HOTPOT_OFFICIAL_URL = "http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json"
HOTPOT_MIRROR_URL = (
    "https://huggingface.co/datasets/RAGLAB/data/resolve/main/"
    "eval_datasets/HotPotQA/hotpot_dev_distractor_v1.json?download=true"
)
HOTPOT_MIRROR_SHA256 = "4e9ecb5c8d3b719f624d66b60f8d56bf227f03914f5f0753d6fa1b359d7104ea"
HOTPOT_OFFICIAL_PATH = ROOT / "hotpot_dev_distractor_v1.json"
HOTPOT_SOURCE_MANIFEST = ROOT / "hotpot_dev_source_manifest.json"

def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def sha256_text(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def membership_sha(ids):
    return sha256_text("\n".join(map(str, ids)) + "\n")

def norm_q(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    return re.sub(r"\s+", " ", s).strip()

print("ARC-v0.37 preflight initialized.")

In [ ]:
# Cell 2 — Download official BEIR HotpotQA retrieval benchmark
if not (BEIR_DIR / "queries.jsonl").is_file():
    if not BEIR_ZIP.is_file():
        print("Downloading BEIR HotpotQA...")
        with requests.get(BEIR_URL, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(BEIR_ZIP, "wb") as f:
                for chunk in r.iter_content(8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
    print("Extracting BEIR HotpotQA...")
    with zipfile.ZipFile(BEIR_ZIP) as z:
        z.extractall(ROOT)

assert (BEIR_DIR / "queries.jsonl").is_file()
assert (BEIR_DIR / "qrels" / "test.tsv").is_file()
assert (BEIR_DIR / "qrels" / "dev.tsv").is_file()

print("BEIR ZIP SHA256:", sha256_file(BEIR_ZIP))
print("BEIR source ready.")

In [ ]:
# Cell 3 — Acquire HotpotQA dev distractor with official→pinned-mirror fallback
#
# Benchmark identity remains HotpotQA dev distractor v1.
# The mirror is transport-only and is SHA256-pinned.

def download_stream(url, dest, connect_timeout, read_timeout):
    tmp = Path(str(dest) + ".part")
    if tmp.exists():
        tmp.unlink()
    with requests.get(
        url,
        stream=True,
        timeout=(connect_timeout, read_timeout),
        allow_redirects=True,
        headers={"User-Agent": "Mozilla/5.0 ARC-v0.37.1 research audit"},
    ) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(4 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
    tmp.replace(dest)

transport = None
official_error = None

if not HOTPOT_OFFICIAL_PATH.is_file():
    print("Trying official HotpotQA CMU host...")
    try:
        download_stream(
            HOTPOT_OFFICIAL_URL,
            HOTPOT_OFFICIAL_PATH,
            connect_timeout=20,
            read_timeout=180,
        )
        transport = "official_cmu"
        print("Official download succeeded.")
    except Exception as e:
        official_error = repr(e)
        print("Official host unavailable:", official_error)
        print("Falling back to pinned RAGLAB/Hugging Face mirror...")
        download_stream(
            HOTPOT_MIRROR_URL,
            HOTPOT_OFFICIAL_PATH,
            connect_timeout=30,
            read_timeout=300,
        )
        transport = "raglab_huggingface_mirror"
        got_sha = sha256_file(HOTPOT_OFFICIAL_PATH)
        print("Mirror SHA256:", got_sha)
        assert got_sha == HOTPOT_MIRROR_SHA256, (
            "Pinned mirror SHA256 mismatch. STOP; do not use these bytes."
        )
else:
    transport = "preexisting_local_file"
    print("Reusing local HotpotQA dev file:", HOTPOT_OFFICIAL_PATH)

OFFICIAL = json.loads(HOTPOT_OFFICIAL_PATH.read_text(encoding="utf-8"))
assert isinstance(OFFICIAL, list)
assert len(OFFICIAL) == N_OFFICIAL_EXPECTED, (
    f"Expected {N_OFFICIAL_EXPECTED} HotpotQA dev examples, got {len(OFFICIAL)}"
)

required = {"_id", "question", "answer"}
missing_fields = [
    (i, sorted(required - set(ex)))
    for i, ex in enumerate(OFFICIAL)
    if not required.issubset(ex)
]
assert not missing_fields, f"Examples missing required fields: {missing_fields[:5]}"

official_by_id = {str(ex["_id"]): ex for ex in OFFICIAL}
assert len(official_by_id) == N_OFFICIAL_EXPECTED, "Duplicate HotpotQA _id values."

source_manifest = {
    "study_id": "ARC-v0.37.1",
    "benchmark_identity": "HotpotQA dev distractor v1",
    "transport": transport,
    "official_url": HOTPOT_OFFICIAL_URL,
    "mirror_url": HOTPOT_MIRROR_URL,
    "mirror_expected_sha256": HOTPOT_MIRROR_SHA256,
    "official_attempt_error": official_error,
    "local_bytes": HOTPOT_OFFICIAL_PATH.stat().st_size,
    "local_sha256": sha256_file(HOTPOT_OFFICIAL_PATH),
    "n_examples": len(OFFICIAL),
    "no_outcomes_before_transport_fix": True,
}
HOTPOT_SOURCE_MANIFEST.write_text(
    json.dumps(source_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("HotpotQA examples:", len(OFFICIAL))
print("Transport:", transport)
print("Local SHA256:", sha256_file(HOTPOT_OFFICIAL_PATH))
print("Native answer examples:", [(x["_id"], x["answer"]) for x in OFFICIAL[:3]])
print("HOTPOTQA SOURCE ACQUISITION — PASS")

In [ ]:
# Cell 4 — Load BEIR query IDs and qrels split membership
BEIR_QUERY = {}
with open(BEIR_DIR / "queries.jsonl", encoding="utf-8") as f:
    for line in f:
        o = json.loads(line)
        BEIR_QUERY[str(o["_id"])] = str(o.get("text", ""))

def qrel_query_ids(path):
    ids = set()
    with open(path, encoding="utf-8") as f:
        header = f.readline()
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                ids.add(str(parts[0]))
    return ids

TEST_IDS = qrel_query_ids(BEIR_DIR / "qrels" / "test.tsv")
DEV_IDS = qrel_query_ids(BEIR_DIR / "qrels" / "dev.tsv")

print("BEIR query rows:", len(BEIR_QUERY))
print("BEIR qrels test queries:", len(TEST_IDS))
print("BEIR qrels dev queries:", len(DEV_IDS))

assert all(q in BEIR_QUERY for q in TEST_IDS)

In [ ]:
# Cell 5 — HARD native-gold lineage audit
#
# Primary criterion:
#   BEIR test qrels IDs == official HotpotQA dev IDs
#
# Secondary criterion:
#   exact normalized question text also agrees for every ID.
#
# No fuzzy matching is used.

OFFICIAL_IDS = set(official_by_id)
only_beir = sorted(TEST_IDS - OFFICIAL_IDS)
only_official = sorted(OFFICIAL_IDS - TEST_IDS)

print("Official IDs:", len(OFFICIAL_IDS))
print("BEIR test IDs:", len(TEST_IDS))
print("Only BEIR:", len(only_beir))
print("Only official:", len(only_official))

if only_beir:
    print("Only-BEIR examples:", only_beir[:20])
if only_official:
    print("Only-official examples:", only_official[:20])

assert TEST_IDS == OFFICIAL_IDS, (
    "BEIR HotpotQA test IDs are not exactly the official 7,405-question dev IDs. "
    "STOP before any benchmark freeze or answer generation."
)

text_mismatch = []
for qid in sorted(TEST_IDS):
    a = norm_q(BEIR_QUERY[qid])
    b = norm_q(official_by_id[qid]["question"])
    if a != b:
        text_mismatch.append((qid, BEIR_QUERY[qid], official_by_id[qid]["question"]))

print("Question-text mismatches:", len(text_mismatch))
if text_mismatch:
    print("Mismatch examples:", text_mismatch[:20])

assert not text_mismatch, (
    "IDs align but question text differs. STOP and audit preprocessing before freezing."
)

empty_answers = [
    qid for qid in sorted(TEST_IDS)
    if not str(official_by_id[qid].get("answer", "")).strip()
]
assert not empty_answers, f"Official HotpotQA examples with empty answers: {empty_answers[:20]}"

print("HOTPOTQA NATIVE-GOLD ALIGNMENT — 7405/7405 PASS")
print("Transport provenance:", json.loads(HOTPOT_SOURCE_MANIFEST.read_text(encoding="utf-8"))["transport"])

In [ ]:
# Cell 6 — Prospective FIT / VALID split by frozen hash salt
#
# Split is independent of retrieval and answer outcomes.

def salted_rank(qid, salt):
    return hashlib.sha256(f"{salt}\n{qid}".encode("utf-8")).hexdigest()

ALL = sorted(TEST_IDS)
ORDERED = sorted(ALL, key=lambda q: (salted_rank(q, SPLIT_SALT), q))

mid = len(ORDERED) // 2
FIT_IDS = sorted(ORDERED[:mid])
VALID_IDS = sorted(ORDERED[mid:])

assert not (set(FIT_IDS) & set(VALID_IDS))
assert set(FIT_IDS) | set(VALID_IDS) == set(ALL)

print("FIT:", len(FIT_IDS))
print("VALID:", len(VALID_IDS))
print("FIT SHA:", membership_sha(FIT_IDS))
print("VALID SHA:", membership_sha(VALID_IDS))

In [ ]:
# Cell 7 — Freeze untouched 500-query downstream MAIN subset inside VALID
#
# This subset is selected before ANN calibration and before any answer outcome.

MAIN_IDS = sorted(
    VALID_IDS,
    key=lambda q: (salted_rank(q, MAIN_SALT), q)
)[:N_MAIN]
MAIN_IDS = sorted(MAIN_IDS)

assert len(MAIN_IDS) == N_MAIN
assert set(MAIN_IDS).issubset(VALID_IDS)

# Gold answers are native fields from the official benchmark.
GOLD = {qid: str(official_by_id[qid]["answer"]) for qid in MAIN_IDS}

# Optional descriptive metadata only; never used for inclusion.
TYPE_COUNTS = defaultdict(int)
LEVEL_COUNTS = defaultdict(int)
for qid in MAIN_IDS:
    TYPE_COUNTS[str(official_by_id[qid].get("type", "unknown"))] += 1
    LEVEL_COUNTS[str(official_by_id[qid].get("level", "unknown"))] += 1

print("Main queries:", len(MAIN_IDS))
print("Main membership SHA:", membership_sha(MAIN_IDS))
print("Type counts:", dict(TYPE_COUNTS))
print("Level counts:", dict(LEVEL_COUNTS))
print("Gold preview:", [(q, GOLD[q]) for q in MAIN_IDS[:5]])

In [ ]:
# Cell 8 — Freeze prospective ARC-v0.37 benchmark protocol BEFORE outcomes

POLICIES = [
    {"alpha": 0.1, "feedback": "mean-k20"},
    {"alpha": 0.1, "feedback": "softmax-k20-tau0.1"},
    {"alpha": 0.3, "feedback": "mean-k20"},
    {"alpha": 0.3, "feedback": "softmax-k20-tau0.1"},
    {"alpha": 0.5, "feedback": "mean-k20"},
    {"alpha": 0.5, "feedback": "softmax-k20-tau0.1"},
    {"alpha": 0.7, "feedback": "mean-k20"},
    {"alpha": 0.7, "feedback": "softmax-k20-tau0.1"},
]

PROTOCOL = {
    "study_id": "ARC-v0.37.1",
    "title": "HotpotQA Native-Gold Downstream QA Consequence Audit",
    "status": "FROZEN_BEFORE_RETRIEVAL_OR_ANSWER_OUTCOMES",
    "scientific_role": "new independent benchmark with native QA gold; v0.37.1 transport-only repair before outcomes",
    "dataset": {
        "retrieval": "BEIR HotpotQA test / global 5.23M corpus",
        "qa_gold": "Official HotpotQA dev distractor",
        "alignment": "exact shared _id plus normalized exact question text",
        "n_full": len(ALL),
        "fit_n": len(FIT_IDS),
        "valid_n": len(VALID_IDS),
        "main_n": len(MAIN_IDS),
        "fit_sha256": membership_sha(FIT_IDS),
        "valid_sha256": membership_sha(VALID_IDS),
        "main_sha256": membership_sha(MAIN_IDS),
        "split_salt": SPLIT_SALT,
        "main_salt": MAIN_SALT,
    },
    "retrieval_design": {
        "encoder": "thenlper/gte-small",
        "dimension": 384,
        "normalized_embeddings": True,
        "nlist": 4096,
        "representation_low": "IVF-PQ32 @ nprobe64",
        "shared_high": "IVF-SQ8 @ nprobe64",
        "search_effort_low": "IVF-SQ8 @ FIT-selected nprobe",
        "search_grid": [1, 2, 4, 8, 16, 32],
        "fit_calibration_target": "absolute match to representation one-shot nDCG@10 loss",
        "valid_retuning": False,
        "top_retrieve": 100,
    },
    "feedback": {
        "operator": "anchored centroid",
        "H": 4,
        "policies": POLICIES,
        "feedback_k": 20,
    },
    "answerer": {
        "model": "Qwen/Qwen2.5-3B-Instruct",
        "revision": "resolve-and-pin-before-first-generation",
        "decoding": "deterministic",
        "evidence_k": 5,
        "passage_serialization": "title + text",
        "max_chars_per_passage": 850,
        "ann_scores_exposed": False,
        "max_new_tokens": 32,
    },
    "answer_metrics": {
        "primary_metric": "HotpotQA-style normalized token F1",
        "secondary": ["HotpotQA-style exact match"],
        "gold": "single official answer string per _id",
    },
    "primary": {
        "D0_q": "F1_search_low_t0 - F1_rep_low_t0",
        "DH_q": "mean_policy(F1_search_low_H - F1_rep_low_H)",
        "estimand": "mean_query(DH_q - D0_q)",
        "name": "feedback_added_F1_mechanism_contrast",
        "classification": {
            "positive": "95% query-bootstrap CI strictly > 0",
            "reversed": "95% query-bootstrap CI strictly < 0",
            "unresolved": "95% CI includes 0",
        },
    },
    "statistics": {
        "independent_unit": "query",
        "bootstrap_reps": 10000,
        "seed": SEED,
    },
    "retention_rule": (
        "Retain positive, null, or reversed outcomes. "
        "No MAIN membership, answer model, prompt, evidence depth, horizon, policy, "
        "metric, or primary-estimand tuning after freeze."
    ),
    "transport_provenance": {
        "manifest": json.loads(HOTPOT_SOURCE_MANIFEST.read_text(encoding="utf-8")),
        "transport_fix_only": True,
        "scientific_choices_changed": False,
        "outcomes_seen_before_fix": False,
    },
    "source_hashes": {
        "beir_zip": sha256_file(BEIR_ZIP),
        "official_hotpot_dev": sha256_file(HOTPOT_OFFICIAL_PATH),
    },
}

PROTOCOL_PATH = OUT / "V0371_FROZEN_PROTOCOL.json"
PROTOCOL_PATH.write_text(
    json.dumps(PROTOCOL, indent=2, sort_keys=True, ensure_ascii=False),
    encoding="utf-8",
)
PROTOCOL_SHA = sha256_file(PROTOCOL_PATH)
(OUT / "V0371_PROTOCOL_SHA256.txt").write_text(PROTOCOL_SHA + "\n", encoding="utf-8")

(OUT / "V0371_FIT_IDS.txt").write_text("\n".join(FIT_IDS) + "\n", encoding="utf-8")
(OUT / "V0371_VALID_IDS.txt").write_text("\n".join(VALID_IDS) + "\n", encoding="utf-8")
(OUT / "V0371_MAIN500_IDS.txt").write_text("\n".join(MAIN_IDS) + "\n", encoding="utf-8")
(OUT / "V0371_MAIN500_GOLD.json").write_text(
    json.dumps(GOLD, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("Protocol SHA:", PROTOCOL_SHA)
print("ARC-v0.37.1 BENCHMARK FREEZE — PASS")
print("STOP HERE. Commit the executed freeze artifacts before retrieval/indexing.")

## Required stop point

Do **not** build embeddings, calibrate `nprobe`, reconstruct trajectories, or run Qwen.

Required success sequence:

1. `HOTPOTQA SOURCE ACQUISITION — PASS`
2. `HOTPOTQA NATIVE-GOLD ALIGNMENT — 7405/7405 PASS`
3. `ARC-v0.37.1 BENCHMARK FREEZE — PASS`

Then save and commit the executed notebook, source manifest, frozen protocol,
protocol SHA, split IDs, MAIN500 IDs, and MAIN500 gold **before** any retrieval
calibration or answer generation.

Recommended commit message:

`Freeze ARC v0.37.1 HotpotQA native-gold benchmark after transport fix`